In [1]:
import json
from sklearn.model_selection import train_test_split

In [ ]:
# with open('Data/data_augmented.json', 'r') as f:
#     data = json.load(f)
# train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)
# train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

# with open('Data/train_data.json', 'w') as f:
#     json.dump(train_data, f, indent=4)
# with open('Data/val_data.json', 'w') as f:
#     json.dump(val_data, f, indent=4)
# with open('Data/test_data.json', 'w') as f:
#     json.dump(test_data, f, indent=4)

In [ ]:
# len(data)

46143

In [2]:
with open('Data/train_data.json', 'r') as f:
    train_data = json.load(f)
with open('Data/val_data.json', 'r') as f:
    val_data = json.load(f)
with open('Data/test_data.json', 'r') as f:
    test_data = json.load(f)

In [6]:
test_data.sort(key=lambda x: len(x['case_details']), reverse=True)
test_data[2]

{'CNR': 'HCBM010062762017',
 'bail_type': 'anticipatory-bail',
 'age_available': False,
 'ages': None,
 'health_condition': 'none.',
 'past_criminal_record_exists': False,
 'past_criminal_record_charges': None,
 'statutes': ['328 IPC',
  '504 IPC',
  '164 CrPC',
  '10 the protection of children from sexual offences act, 2012',
  '506 IPC',
  '376 IPC',
  '4 the protection of children from sexual offences act, 2012',
  '8 the protection of children from sexual offences act, 2012'],
 'case_details': "The applicant is arrested on 12/7/2016 in Crime No. 319 of 2016 registered at V.P. Road Police Station on 11/7/2016. The investigation is completed and charge-sheet is filed against the applicant for offence punishable under section 376, 328, 504, 506 of the Indian Penal Code and under section 4, 8, 10 of the Protection of Children from Sexual Offenses Act, 2012. The applicant is the biological father of the victim Ms. X. On 11/7/2016 Ms. X lodged a report at the police station that in the y

In [5]:
def prep_data_for_ft(data, filename, include_statute_details=False): 
    with open('Data/' + filename + '.jsonl', 'w', encoding='utf-8') as f:
        for item in data:
            bail_type = "Applicant applied for " + item['bail_type'] + ". "
            age = "The age of the applicant/s is/are " + str(item['ages']) + ". " if item['age_available'] else "Age not available. "
            health = "The health condition of the applicant is " + item['health_condition'] + " " if item['health_condition'] is not None else "none. "
            past = "Past criminal records for the applicant "+ ("exist." if item['past_criminal_record_exists'] else "do not exist.") + " "
            statutes = "The relevant statutes are: " + ", ".join(item['statutes']) + ". "
            custody = "The applicant is in custody for " + str(item['days_in_custody']) + " days. " if item['days_in_custody'] is not None else ""
            case_details = item['case_details'] + " " if item['case_details'] is not None else ""
            outcome = "The outcome of the case is " + item['outcome'] + "."
            reasoning = item['reasoning'] if item['reasoning'] is not None else ""
            statutes_info = " ".join(item['statute_details']) + " " if item['statute_details'] is not None else ""

            user_input = bail_type + age + health + past + statutes + custody + case_details + outcome + (" " + statutes_info if include_statute_details else "")
            output = reasoning

            record = {
                "messages": [
                    {"role": "user", "content": user_input.strip()},
                    {"role": "assistant", "content": output.strip()}
                ]
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

In [6]:
prep_data_for_ft(train_data, 'train_ft_with_statutes', include_statute_details=True)
prep_data_for_ft(val_data, 'val_ft_with_statutes', include_statute_details=True)
prep_data_for_ft(test_data, 'test_ft_with_statutes', include_statute_details=True)

In [8]:
import json

with open("Data/train_ft_with_statutes.jsonl") as f:
    for i, line in enumerate(f, 1):
        try:
            obj = json.loads(line)
            assert "messages" in obj
            assert len(obj["messages"]) >= 2
        except Exception as e:
            print(f"Error on line {i}: {e}")